# Section 8 — Multimodal Feature Extraction

## Task 8.1 — Environment Setup and Google Drive Mounting

**Objective:** Configure Google Colab for GPU-accelerated feature extraction using pretrained DistilBERT and CLIP ViT-B/32. Verify the Google Drive project folder and preprocessing artifacts without loading the large image archive into RAM.

**Before running:** In Colab, select **Runtime → Change runtime type → T4 GPU** (if available). This notebook does not load `images.npz` or start feature extraction.


In [ ]:
import torch
import numpy as np
from pathlib import Path
from google.colab import drive

# Mount Google Drive (follow the sign-in/authorization prompt)
drive.mount("/content/drive")

# Folder created on Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/Multi Modal - Project 2.2")

# Confirm compute environment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
else:
    print("GPU not available. Change Runtime type to T4 GPU if offered.")

# Check file presence only (do not load large files)
print("\nProject directory:", PROJECT_DIR)
print("Project directory exists:", PROJECT_DIR.is_dir())
required_files = [
    "data.npy",
    "images.npz",
    "split_indices.npz",
    "group_ids.npy",
    "cv_fold_indices.npz",
]
print("\nProject files:")
for filename in required_files:
    filepath = PROJECT_DIR / filename
    print(f"{filename}: {'OK' if filepath.is_file() else 'MISSING'}")


Mounted at /content/drive
PyTorch version: 2.14.1+cu130
NumPy version: 2.4.6
Device: cuda
GPU: Tesla T4
GPU memory (GiB): 14.56

Project directory: /content/drive/MyDrive/Multi Modal - Project 2.2
Project directory exists: True

Project files:
data.npy: MISSING
images.npz: MISSING
split_indices.npz: OK
group_ids.npy: OK
cv_fold_indices.npz: OK


### Interpretation / Next step

If the notebook prints **Device: cuda** and the three preprocessing artifacts (`split_indices.npz`, `group_ids.npy`, and `cv_fold_indices.npz`) show **OK**, the environment setup is working. `data.npy` and `images.npz` may still show **MISSING** while they are uploading to Drive. This is not an error in the notebook.

**Next task:** Validate the saved indices and dataset metadata without loading all poster images into memory, then begin text feature extraction.


### 8.2. Load and Validate Cross-Validation Indices

This task loads the saved duplicate-group IDs and
cross-validation folds, verifies dataset partitioning,
and checks for group leakage.

The independent test set is preserved for final evaluation.

In [ ]:
import numpy as np

# Load saved duplicate-group IDs
group_ids = np.load(PROJECT_DIR / "group_ids.npy")

# Load saved cross-validation indices
with np.load(PROJECT_DIR / "cv_fold_indices.npz") as cv:
    dev_idx = cv["dev_idx"]
    test_idx = cv["test_idx"]

    cv_folds = []

    for fold in range(1, 6):
        train_idx = cv[f"fold_{fold}_train_idx"]
        val_idx = cv[f"fold_{fold}_val_idx"]

        cv_folds.append((train_idx, val_idx))

# Verify dataset partitioning
assert len(group_ids) == 25959
assert len(dev_idx) == 18176
assert len(test_idx) == 7783

assert len(np.unique(np.concatenate(
    [dev_idx, test_idx]
))) == 25959

# Verify development-test group isolation
assert np.intersect1d(
    group_ids[dev_idx],
    group_ids[test_idx]
).size == 0

print("Cross-validation validation results:\n")

all_val_indices = []

for fold, (train_idx, val_idx) in enumerate(cv_folds, 1):

    overlap = np.intersect1d(
        group_ids[train_idx],
        group_ids[val_idx]
    ).size

    assert overlap == 0
    assert np.array_equal(
        np.sort(np.concatenate([train_idx, val_idx])),
        dev_idx
    )

    all_val_indices.append(val_idx)

    print(
        f"Fold {fold}: "
        f"Train={len(train_idx)}, "
        f"Validation={len(val_idx)}, "
        f"Group overlap={overlap}"
    )

assert np.array_equal(
    np.sort(np.concatenate(all_val_indices)),
    dev_idx
)

print("\nAll cross-validation folds verified.")
print("Development records:", len(dev_idx))
print("Independent test records:", len(test_idx))

Cross-validation validation results:

Fold 1: Train=14540, Validation=3636, Group overlap=0
Fold 2: Train=14541, Validation=3635, Group overlap=0
Fold 3: Train=14541, Validation=3635, Group overlap=0
Fold 4: Train=14541, Validation=3635, Group overlap=0
Fold 5: Train=14541, Validation=3635, Group overlap=0

All cross-validation folds verified.
Development records: 18176
Independent test records: 7783


### 8.3. DistilBERT Text Feature Extraction

#### 8.3.1. Load Movie Records and Plot Summaries

This task loads the movie records and extracts plot
summaries and multi-label genre targets.

The text data will be processed using a pretrained
DistilBERT encoder to obtain 768-dimensional embeddings.

The original record order is preserved to maintain
alignment with the saved cross-validation indices
and poster images.

In [ ]:
import numpy as np

# Check whether the dataset has finished uploading
data_path = PROJECT_DIR / "data.npy"

assert data_path.exists(), (
    "data.npy is not available yet. "
    "Please check the Google Drive upload."
)

# Load movie records
records = np.load(data_path, allow_pickle=True)

# Extract plot summaries and genre labels
summaries = [str(text) for text in records[:, 2]]
labels = np.stack(records[:, 1]).astype(np.int64)

# Verify dataset structure
assert len(records) == 25959
assert len(summaries) == 25959
assert labels.shape == (25959, 23)
assert all(summary.strip() for summary in summaries)

# Verify alignment with CV indices
assert max(np.concatenate([dev_idx, test_idx])) < len(records)

print("Movie records loaded successfully.")
print("Records shape:", records.shape)
print("Total plot summaries:", len(summaries))
print("Genre labels shape:", labels.shape)
print("Number of genres:", labels.shape[1])
print("Development records:", len(dev_idx))
print("Test records:", len(test_idx))
print("\nText data preparation: PASSED")

Movie records loaded successfully.
Records shape: (25959, 3)
Total plot summaries: 25959
Genre labels shape: (25959, 23)
Number of genres: 23
Development records: 18176
Test records: 7783

Text data preparation: PASSED


#### 8.3.2. Load Pretrained DistilBERT Tokenizer and Encoder

A pretrained DistilBERT-base-uncased model is used to
extract contextual representations from movie plot summaries.

The encoder is frozen and operates in evaluation mode,
allowing efficient feature extraction without fine-tuning.

Text inputs are truncated to a maximum of 256 tokens,
consistent with the preprocessing design in Project 2.1.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# Model configuration
MODEL_NAME = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 256

# Load pretrained tokenizer and encoder
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

text_encoder = AutoModel.from_pretrained(MODEL_NAME)
text_encoder = text_encoder.to(device)

# Freeze pretrained encoder
text_encoder.requires_grad_(False)
text_encoder.eval()

# Verify model configuration
print("Model:", MODEL_NAME)
print("Device:", next(text_encoder.parameters()).device)
print("Hidden size:", text_encoder.config.dim)
print("Maximum token length:", MAX_LENGTH)
print("Encoder frozen:", not any(
    p.requires_grad for p in text_encoder.parameters()
))

print("\nDistilBERT loaded successfully.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: distilbert/distilbert-base-uncased
Device: cuda:0
Hidden size: 768
Maximum token length: 256
Encoder frozen: True

DistilBERT loaded successfully.


#### 8.3.3. Test DistilBERT Feature Extraction

This task tests the frozen DistilBERT encoder on
eight movie plot summaries before full-dataset extraction.

The first-token hidden representation is extracted
as a 768-dimensional text embedding for each movie.

The resulting embeddings are checked for shape
consistency and invalid numerical values.

In [ ]:
import numpy as np
import torch

# Select 8 sample plot summaries
sample_texts = summaries[:8]

# Tokenize sample texts
inputs = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt"
)

# Move inputs to GPU
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

# Extract text embeddings
with torch.inference_mode():
    outputs = text_encoder(**inputs)
    sample_embeddings = outputs.last_hidden_state[:, 0, :]

# Convert to NumPy
sample_embeddings = sample_embeddings.cpu().numpy()

# Validate embeddings
assert sample_embeddings.shape == (8, 768)
assert np.isfinite(sample_embeddings).all()

print("Input batch size:", len(sample_texts))
print("Token IDs shape:", tuple(inputs["input_ids"].shape))
print("Embeddings shape:", sample_embeddings.shape)
print("Contains NaN:", np.isnan(sample_embeddings).any())
print("Contains Inf:", np.isinf(sample_embeddings).any())

print("\nDistilBERT sample extraction: PASSED")

Input batch size: 8
Token IDs shape: (8, 256)
Embeddings shape: (8, 768)
Contains NaN: False
Contains Inf: False

DistilBERT sample extraction: PASSED


#### 8.3.4. Full-Dataset DistilBERT Feature Extraction

The frozen DistilBERT encoder is applied to all 25,959
plot summaries using GPU-accelerated mini-batches.

A 768-dimensional first-token embedding is extracted
for each movie. Results are saved in separate chunks
to support checkpointing and recovery from interruptions.

The original record order is preserved to ensure
alignment with poster images and cross-validation indices.

In [ ]:
import numpy as np
import torch
from tqdm.auto import tqdm

# Configuration
BATCH_SIZE = 32
CHUNK_SIZE = 1024
N_MOVIES = len(summaries)
EMBED_DIM = 768

# Create output directory
CHUNK_DIR = PROJECT_DIR / "text_chunks_distilbert_cls_256"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

text_encoder.eval()

print("Starting DistilBERT feature extraction...")
print("Total movies:", N_MOVIES)
print("Batch size:", BATCH_SIZE)
print("Chunk size:", CHUNK_SIZE)
print("Device:", device)

# Extract embeddings in chunks
for start in tqdm(
    range(0, N_MOVIES, CHUNK_SIZE),
    desc="Text extraction"
):
    end = min(start + CHUNK_SIZE, N_MOVIES)

    output_file = CHUNK_DIR / f"text_{start:05d}_{end:05d}.npy"
    expected_shape = (end - start, EMBED_DIM)

    # Skip valid completed chunks
    if output_file.exists():
        try:
            saved = np.load(output_file, mmap_mode="r")
            valid = (
                saved.shape == expected_shape
                and saved.dtype == np.float32
                and np.isfinite(saved).all()
            )
            del saved

            if valid:
                continue
        except (OSError, ValueError):
            pass

    chunk_embeddings = []

    # Process mini-batches
    for batch_start in range(start, end, BATCH_SIZE):

        batch_end = min(batch_start + BATCH_SIZE, end)
        batch_texts = summaries[batch_start:batch_end]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(device)

        with torch.inference_mode():
            outputs = text_encoder(**inputs)

            embeddings = (
                outputs.last_hidden_state[:, 0, :]
                .float()
                .cpu()
                .numpy()
            )

        chunk_embeddings.append(embeddings)

    # Combine mini-batches
    chunk_array = np.concatenate(
        chunk_embeddings, axis=0
    ).astype(np.float32)

    # Verify chunk
    assert chunk_array.shape == expected_shape
    assert np.isfinite(chunk_array).all()

    # Save safely
    temp_file = output_file.with_suffix(".tmp.npy")
    np.save(temp_file, chunk_array)
    temp_file.replace(output_file)

print("\nFull text extraction completed.")
print("Total movies:", N_MOVIES)
print("Saved directory:", CHUNK_DIR)

Starting DistilBERT feature extraction...
Total movies: 25959
Batch size: 32
Chunk size: 1024
Device: cuda


Text extraction:   0%|          | 0/26 [00:00<?, ?it/s]


Full text extraction completed.
Total movies: 25959
Saved directory: /content/drive/MyDrive/Multi Modal - Project 2.2/text_chunks_distilbert_cls_256


#### 8.3.5. Combine and Validate Text Embeddings

The extracted DistilBERT embedding chunks are combined
into a single feature matrix while preserving the
original movie record order.

The final embeddings are validated for dimensional
consistency, data type, and invalid numerical values.

The resulting feature matrix is cached for subsequent
unimodal and multimodal classification experiments.

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# Configuration
N_MOVIES = len(summaries)
EMBED_DIM = 768
CHUNK_SIZE = 1024

# Output paths
OUTPUT_PATH = PROJECT_DIR / "text_embeddings.npy"
TEMP_PATH = PROJECT_DIR / "text_embeddings.partial.npy"

# Create the final embedding matrix on disk
text_embeddings = np.lib.format.open_memmap(
    TEMP_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(N_MOVIES, EMBED_DIM)
)

print("Combining DistilBERT embedding chunks...")

# Load and combine saved chunks
for start in tqdm(
    range(0, N_MOVIES, CHUNK_SIZE),
    desc="Combining chunks"
):
    end = min(start + CHUNK_SIZE, N_MOVIES)

    chunk_path = (
        CHUNK_DIR / f"text_{start:05d}_{end:05d}.npy"
    )

    assert chunk_path.exists(), f"Missing chunk: {chunk_path}"

    chunk = np.load(chunk_path, mmap_mode="r")

    # Validate chunk
    assert chunk.shape == (end - start, EMBED_DIM)
    assert chunk.dtype == np.float32
    assert np.isfinite(chunk).all()

    # Preserve original record order
    text_embeddings[start:end] = chunk

    del chunk

# Flush data to disk
text_embeddings.flush()
del text_embeddings

# Validate the combined file
saved_embeddings = np.load(TEMP_PATH, mmap_mode="r")

assert saved_embeddings.shape == (25959, 768)
assert saved_embeddings.dtype == np.float32
assert np.isfinite(saved_embeddings).all()

print("\nFinal Embedding Validation:")
print("Shape:", saved_embeddings.shape)
print("Data type:", saved_embeddings.dtype)
print("Contains NaN:", np.isnan(saved_embeddings).any())
print("Contains Inf:", np.isinf(saved_embeddings).any())

del saved_embeddings

# Finalise the output
TEMP_PATH.replace(OUTPUT_PATH)

print("\nText embeddings saved successfully.")
print("Saved to:", OUTPUT_PATH)
print("File size (MiB):", round(
    OUTPUT_PATH.stat().st_size / (1024 ** 2), 2
))
print("Text feature extraction: COMPLETED")

Combining DistilBERT embedding chunks...


Combining chunks:   0%|          | 0/26 [00:00<?, ?it/s]


Final Embedding Validation:
Shape: (25959, 768)
Data type: float32
Contains NaN: False
Contains Inf: False

Text embeddings saved successfully.
Saved to: /content/drive/MyDrive/Multi Modal - Project 2.2/text_embeddings.npy
File size (MiB): 76.05
Text feature extraction: COMPLETED


### 8.4. CLIP Image Feature Extraction

#### 8.4.1. Verify Image Dataset Availability

This task verifies the availability and structure of
the movie poster dataset before feature extraction.

Poster images will be accessed using memory mapping
to minimise RAM consumption.

In [ ]:
from pathlib import Path
import zipfile

IMAGE_PATH = PROJECT_DIR / "images.npz"

print("Checking image dataset...\n")
print("File exists:", IMAGE_PATH.is_file())

if IMAGE_PATH.is_file():
    print(
        "File size (GiB):",
        round(IMAGE_PATH.stat().st_size / (1024**3), 2)
    )

    with zipfile.ZipFile(IMAGE_PATH, "r") as archive:
        print("Archive contents:", archive.namelist())

    print("\nImage dataset verification: PASSED")
else:
    print("\nimages.npz is not available yet.")

Checking image dataset...

File exists: True
File size (GiB): 2.97
Archive contents: ['images.npy']

Image dataset verification: PASSED


#### 8.4.2. Prepare Memory-Mapped Poster Data

The poster array is extracted from the NPZ archive
to a local NPY file using streaming I/O.

Memory mapping enables batch-wise image access without
loading the entire image dataset into RAM.

In [ ]:
import os
import shutil
import zipfile
import numpy as np
from pathlib import Path

# Store the unpacked array on Colab's local disk
POSTER_PATH = Path("/content/posters_uint8.npy")
TEMP_PATH = Path("/content/posters_uint8.partial.npy")

assert IMAGE_PATH.is_file(), "images.npz is missing."

if not POSTER_PATH.exists():

    with zipfile.ZipFile(IMAGE_PATH, "r") as archive:

        assert "images.npy" in archive.namelist()

        required_space = archive.getinfo("images.npy").file_size
        available_space = shutil.disk_usage("/content").free

        assert available_space > required_space + 1024**3, (
            "Insufficient local disk space."
        )

        print("Extracting poster array to local disk...")

        # Stream data without loading the full array into RAM
        with archive.open("images.npy") as source:
            with open(TEMP_PATH, "wb") as destination:
                shutil.copyfileobj(
                    source,
                    destination,
                    length=8 * 1024 * 1024
                )

        os.replace(TEMP_PATH, POSTER_PATH)

# Load using memory mapping
posters = np.load(POSTER_PATH, mmap_mode="r")

# Validate dataset
assert posters.shape == (25959, 3, 256, 160)
assert posters.dtype == np.uint8

print("\nPoster data prepared successfully.")
print("Shape:", posters.shape)
print("Data type:", posters.dtype)
print("Memory mapped:", isinstance(posters, np.memmap))
print("File:", POSTER_PATH)

Extracting poster array to local disk...

Poster data prepared successfully.
Shape: (25959, 3, 256, 160)
Data type: uint8
Memory mapped: True
File: /content/posters_uint8.npy


#### 8.4.3. Load Pretrained CLIP Vision Encoder

The pretrained CLIP ViT-B/32 vision encoder is used to
extract visual representations from movie posters.

The encoder is frozen and operated in evaluation mode
to reduce computational cost.

Poster images are resized, center-cropped, and normalised
using the pretrained CLIP image processor.

The pooled visual representations are used as
768-dimensional image embeddings for classification.

In [ ]:
import torch
from transformers import CLIPVisionModel, CLIPImageProcessor

# Model configuration
IMAGE_MODEL_NAME = "openai/clip-vit-base-patch32"

# Load pretrained image processor
image_processor = CLIPImageProcessor.from_pretrained(
    IMAGE_MODEL_NAME
)

# Load pretrained CLIP Vision encoder
image_encoder = CLIPVisionModel.from_pretrained(
    IMAGE_MODEL_NAME
)

# Move encoder to GPU
image_encoder = image_encoder.to(device)

# Freeze pretrained encoder
image_encoder.requires_grad_(False)
image_encoder.eval()

# Verify model configuration
print("Model:", IMAGE_MODEL_NAME)
print("Device:", next(image_encoder.parameters()).device)
print("Hidden size:", image_encoder.config.hidden_size)
print("Image size:", image_encoder.config.image_size)

print(
    "Encoder frozen:",
    not any(p.requires_grad for p in image_encoder.parameters())
)

print("\nCLIP Vision encoder loaded successfully.")

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.final_layer_norm.weight        

Model: openai/clip-vit-base-patch32
Device: cuda:0
Hidden size: 768
Image size: 224
Encoder frozen: True

CLIP Vision encoder loaded successfully.


#### 8.4.4. Test CLIP Image Feature Extraction

The frozen CLIP Vision encoder is tested on eight movie
posters before full-dataset feature extraction.

Poster images are converted from channel-first format
to RGB images and processed using the pretrained CLIP
image processor.

The resulting embeddings are validated for shape
consistency and numerical stability.

In [ ]:
import numpy as np
import torch
from PIL import Image

# Select 8 sample posters
sample_images = []

for i in range(8):
    # Convert CHW to HWC
    image_array = np.asarray(posters[i]).transpose(1, 2, 0)

    # Convert to RGB image
    image = Image.fromarray(
        np.ascontiguousarray(image_array)
    ).convert("RGB")

    sample_images.append(image)

# Apply CLIP preprocessing
inputs = image_processor(
    images=sample_images,
    return_tensors="pt"
)

pixel_values = inputs["pixel_values"].to(device)

# Extract image embeddings
with torch.inference_mode():
    outputs = image_encoder(
        pixel_values=pixel_values
    )

    sample_image_embeddings = (
        outputs.pooler_output
        .float()
        .cpu()
        .numpy()
    )

# Validate embeddings
assert sample_image_embeddings.shape == (8, 768)
assert np.isfinite(sample_image_embeddings).all()

print("Input batch size:", len(sample_images))
print("Pixel values shape:", tuple(pixel_values.shape))
print("Embeddings shape:", sample_image_embeddings.shape)

print(
    "Contains NaN:",
    np.isnan(sample_image_embeddings).any()
)

print(
    "Contains Inf:",
    np.isinf(sample_image_embeddings).any()
)

print("\nCLIP sample extraction: PASSED")

Input batch size: 8
Pixel values shape: (8, 3, 224, 224)
Embeddings shape: (8, 768)
Contains NaN: False
Contains Inf: False

CLIP sample extraction: PASSED


#### 8.4.5. Full-Dataset CLIP Image Feature Extraction

The pretrained CLIP ViT-B/32 vision encoder is applied
to all 25,959 movie posters using GPU-accelerated
mini-batches.

Poster images are read from a memory-mapped array and
processed using the pretrained CLIP image processor.

A 768-dimensional pooled visual representation is
extracted for each movie.

Embeddings are saved in chunks to support checkpointing,
recovery from interruptions, and reproducible experiments.

The original record order is preserved to maintain
alignment with text embeddings and genre labels.

In [ ]:
import numpy as np
import torch
import gc
from PIL import Image
from tqdm.auto import tqdm

# Configuration
IMAGE_BATCH_SIZE = 32
IMAGE_CHUNK_SIZE = 1024
N_MOVIES = len(posters)
IMAGE_EMBED_DIM = 768

# Directory for cached image embeddings
IMAGE_CHUNK_DIR = PROJECT_DIR / "image_chunks_clip_vitb32"
IMAGE_CHUNK_DIR.mkdir(parents=True, exist_ok=True)

# Release the previous text encoder if still loaded
if "text_encoder" in globals():
    del text_encoder
    gc.collect()
    torch.cuda.empty_cache()

# Ensure CLIP is in evaluation mode
image_encoder.eval()

print("Starting CLIP image feature extraction...")
print("Total movies:", N_MOVIES)
print("Batch size:", IMAGE_BATCH_SIZE)
print("Chunk size:", IMAGE_CHUNK_SIZE)
print("Device:", device)

# Process the dataset in chunks
for chunk_start in tqdm(
    range(0, N_MOVIES, IMAGE_CHUNK_SIZE),
    desc="Extracting image chunks"
):

    chunk_end = min(
        chunk_start + IMAGE_CHUNK_SIZE,
        N_MOVIES
    )

    chunk_path = IMAGE_CHUNK_DIR / (
        f"image_{chunk_start:05d}_{chunk_end:05d}.npy"
    )

    expected_shape = (
        chunk_end - chunk_start,
        IMAGE_EMBED_DIM
    )

    # Skip previously completed, valid chunks
    if chunk_path.exists():
        try:
            existing = np.load(chunk_path, mmap_mode="r")

            valid = (
                existing.shape == expected_shape
                and existing.dtype == np.float32
                and np.isfinite(existing).all()
            )

            del existing

            if valid:
                continue

        except (OSError, ValueError):
            pass

    batch_embeddings = []

    # Process mini-batches
    for batch_start in range(
        chunk_start,
        chunk_end,
        IMAGE_BATCH_SIZE
    ):

        batch_end = min(
            batch_start + IMAGE_BATCH_SIZE,
            chunk_end
        )

        batch_images = []

        # Load posters without loading the full dataset
        for i in range(batch_start, batch_end):

            image_array = (
                np.asarray(posters[i])
                .transpose(1, 2, 0)
            )

            image = Image.fromarray(
                np.ascontiguousarray(image_array)
            ).convert("RGB")

            batch_images.append(image)

        # Apply pretrained CLIP preprocessing
        inputs = image_processor(
            images=batch_images,
            return_tensors="pt"
        )

        pixel_values = inputs["pixel_values"].to(device)

        # Extract image embeddings
        with torch.inference_mode():

            outputs = image_encoder(
                pixel_values=pixel_values
            )

            embeddings = (
                outputs.pooler_output
                .float()
                .cpu()
                .numpy()
            )

        batch_embeddings.append(embeddings)

        del batch_images, inputs, pixel_values, outputs

    # Combine embeddings within the chunk
    chunk_embeddings = np.concatenate(
        batch_embeddings,
        axis=0
    ).astype(np.float32)

    # Validate extracted features
    assert chunk_embeddings.shape == expected_shape
    assert np.isfinite(chunk_embeddings).all()

    # Save chunk safely
    temp_path = chunk_path.with_suffix(".tmp.npy")

    np.save(temp_path, chunk_embeddings)
    temp_path.replace(chunk_path)

    del batch_embeddings, chunk_embeddings

print("\nFull image extraction completed.")
print("Total movies processed:", N_MOVIES)
print("Chunk directory:", IMAGE_CHUNK_DIR)

Starting CLIP image feature extraction...
Total movies: 25959
Batch size: 32
Chunk size: 1024
Device: cuda


Extracting image chunks:   0%|          | 0/26 [00:00<?, ?it/s]


Full image extraction completed.
Total movies processed: 25959
Chunk directory: /content/drive/MyDrive/Multi Modal - Project 2.2/image_chunks_clip_vitb32


#### 8.4.6. Combine and Validate Image Embeddings

The CLIP image embedding chunks are combined into a
single feature matrix while preserving the original
movie record order.

The final embeddings are validated for dimensional
consistency, data type, and numerical stability.

The resulting feature matrix is cached for subsequent
unimodal and multimodal classification experiments.

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# Configuration
N_MOVIES = 25959
IMAGE_EMBED_DIM = 768
IMAGE_CHUNK_SIZE = 1024

# Output paths
IMAGE_OUTPUT_PATH = PROJECT_DIR / "image_embeddings.npy"
IMAGE_TEMP_PATH = PROJECT_DIR / "image_embeddings.partial.npy"

# Create memory-mapped output matrix
image_embeddings = np.lib.format.open_memmap(
    IMAGE_TEMP_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(N_MOVIES, IMAGE_EMBED_DIM)
)

print("Combining CLIP image embedding chunks...")

# Load and combine chunks in original record order
for start in tqdm(
    range(0, N_MOVIES, IMAGE_CHUNK_SIZE),
    desc="Combining image chunks"
):
    end = min(start + IMAGE_CHUNK_SIZE, N_MOVIES)

    chunk_path = (
        IMAGE_CHUNK_DIR /
        f"image_{start:05d}_{end:05d}.npy"
    )

    assert chunk_path.exists(), f"Missing chunk: {chunk_path}"

    chunk = np.load(chunk_path, mmap_mode="r")

    # Validate chunk
    assert chunk.shape == (end - start, IMAGE_EMBED_DIM)
    assert chunk.dtype == np.float32
    assert np.isfinite(chunk).all()

    # Preserve original record order
    image_embeddings[start:end] = chunk

    del chunk

# Flush data to disk
image_embeddings.flush()
del image_embeddings

# Validate combined embeddings
saved_images = np.load(IMAGE_TEMP_PATH, mmap_mode="r")

assert saved_images.shape == (25959, 768)
assert saved_images.dtype == np.float32
assert np.isfinite(saved_images).all()

print("\nFinal Image Embedding Validation:")
print("Shape:", saved_images.shape)
print("Data type:", saved_images.dtype)
print("Contains NaN:", np.isnan(saved_images).any())
print("Contains Inf:", np.isinf(saved_images).any())

del saved_images

# Finalise output file
IMAGE_TEMP_PATH.replace(IMAGE_OUTPUT_PATH)

print("\nImage embeddings saved successfully.")
print("Saved to:", IMAGE_OUTPUT_PATH)
print(
    "File size (MiB):",
    round(IMAGE_OUTPUT_PATH.stat().st_size / (1024**2), 2)
)

print("Image feature extraction: COMPLETED")

Combining CLIP image embedding chunks...


Combining image chunks:   0%|          | 0/26 [00:00<?, ?it/s]


Final Image Embedding Validation:
Shape: (25959, 768)
Data type: float32
Contains NaN: False
Contains Inf: False

Image embeddings saved successfully.
Saved to: /content/drive/MyDrive/Multi Modal - Project 2.2/image_embeddings.npy
File size (MiB): 76.05
Image feature extraction: COMPLETED


In [ ]:
# Load cached embeddings without copying into RAM
text_features = np.load(
    PROJECT_DIR / "text_embeddings.npy",
    mmap_mode="r"
)

image_features = np.load(
    PROJECT_DIR / "image_embeddings.npy",
    mmap_mode="r"
)

# Verify feature dimensions and record alignment
assert text_features.shape == (25959, 768)
assert image_features.shape == (25959, 768)
assert labels.shape == (25959, 23)

assert np.isfinite(text_features).all()
assert np.isfinite(image_features).all()

print("Text embeddings:", text_features.shape)
print("Image embeddings:", image_features.shape)
print("Genre labels:", labels.shape)

print("\nBoth modalities validated successfully.")
print("Ready for model training.")

Text embeddings: (25959, 768)
Image embeddings: (25959, 768)
Genre labels: (25959, 23)

Both modalities validated successfully.
Ready for model training.
